# DeepSpeed ZeRO-3 Training on Wiki Data with Fault Injection

**Repo:** [ai-dist-training-scaler](https://github.com/TylrDn/ai-dist-training-scaler)  
**Phase:** 3 — Notebook Pipelines

Demonstrates multi-GPU training of a GPT-2 model on the Wiki JSONL dataset
using HuggingFace Accelerate with DeepSpeed ZeRO-3, plus simulated fault injection.

### References
- ZeRO-3: Rajbhandari et al. (2020) https://arxiv.org/abs/1910.02054
- ZeRO-Infinity: Rajbhandari et al. (2021) https://arxiv.org/abs/2104.07857
- HuggingFace Accelerate: https://huggingface.co/docs/accelerate
- AdamW: Loshchilov & Hutter (2019) https://arxiv.org/abs/1711.05101

In [ ]:
from __future__ import annotations

import logging
import os
import random
from dataclasses import dataclass
from pathlib import Path

import torch
import wandb

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

WANDB_DISABLED = os.getenv("WANDB_DISABLED", "false").lower() == "true"


@dataclass
class TrainConfig:
    """Training hyperparameters for ZeRO-3 distributed run."""

    model_name: str = "gpt2"
    wiki_data_path: str = "data/processed/wiki_jsonl"
    output_dir: Path = Path("checkpoints/zero3_wiki")
    deepspeed_config: str = "configs/deepspeed_zero3.json"
    max_steps: int = 1000
    per_device_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float = 3e-4
    max_length: int = 512
    save_every: int = 200
    log_every: int = 10
    # Fault injection: probability of simulating a GPU failure per step
    fault_injection_prob: float = float(os.getenv("FAULT_INJECTION_PROB", "0.0"))


cfg = TrainConfig()
cfg.output_dir.mkdir(parents=True, exist_ok=True)
logger.info("TrainConfig: %s", cfg)

In [ ]:
from accelerate import Accelerator
from datasets import load_dataset
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

# AdamW: Loshchilov & Hutter (2019) https://arxiv.org/abs/1711.05101
# ZeRO-3: Rajbhandari et al. (2020) https://arxiv.org/abs/1910.02054

accelerator = Accelerator(
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    log_with="wandb" if not WANDB_DISABLED else None,
)

if not WANDB_DISABLED and accelerator.is_main_process:
    accelerator.init_trackers(
        project_name="ai-dist-training-scaler",
        config={
            "model": cfg.model_name,
            "lr": cfg.learning_rate,
            "max_steps": cfg.max_steps,
            "batch_size": cfg.per_device_batch_size,
        },
    )

logger.info("Accelerator: %d processes, mixed_precision=%s",
            accelerator.num_processes, accelerator.mixed_precision)

In [ ]:
import glob

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
tokenizer.pad_token = tokenizer.eos_token

jsonl_files = sorted(glob.glob(f"{cfg.wiki_data_path}/shard_*.jsonl"))
if not jsonl_files:
    logger.warning("No shards found — creating dummy dataset for demonstration")
    from datasets import Dataset
    dummy_texts = ["Transformer models are state-of-the-art for NLP tasks."] * 1000
    raw_ds = Dataset.from_dict({"text": dummy_texts})
else:
    raw_ds = load_dataset("json", data_files={"train": jsonl_files}, split="train")


def tokenize(batch: dict) -> dict:
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=cfg.max_length,
        padding="max_length",
    )


cols_to_remove = [c for c in ["text", "id", "meta"] if c in raw_ds.column_names]
ds = raw_ds.map(tokenize, batched=True, remove_columns=cols_to_remove)
ds.set_format(type="torch")
dataloader = DataLoader(ds, batch_size=cfg.per_device_batch_size, shuffle=True)
logger.info("Dataset ready: %d examples", len(ds))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(cfg.model_name)

# AdamW: Loshchilov & Hutter (2019) https://arxiv.org/abs/1711.05101
optimizer = AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=100, num_training_steps=cfg.max_steps
)

model, optimizer, dataloader, scheduler = accelerator.prepare(
    model, optimizer, dataloader, scheduler
)


class FaultInjectionError(RuntimeError):
    """Simulated GPU fault for chaos testing."""


def maybe_inject_fault(prob: float, step: int) -> None:
    """Randomly raise a simulated fault at the given probability per step."""
    if prob > 0 and random.random() < prob:
        raise FaultInjectionError(f"Simulated GPU fault at step {step}")


global_step = 0
model.train()

for batch in dataloader:
    if global_step >= cfg.max_steps:
        break

    with accelerator.accumulate(model):
        try:
            maybe_inject_fault(cfg.fault_injection_prob, global_step)
            outputs = model(input_ids=batch["input_ids"], labels=batch["input_ids"])
            loss = outputs.loss
            accelerator.backward(loss)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        except FaultInjectionError as exc:
            logger.error("Fault injected: %s — resuming from last checkpoint", exc)
            # In a real run, reload the latest checkpoint here
            continue

    global_step += 1

    if global_step % cfg.log_every == 0:
        logger.info("step=%d loss=%.4f", global_step, loss.item())
        if not WANDB_DISABLED:
            accelerator.log({"loss": loss.item(), "step": global_step})

    if global_step % cfg.save_every == 0 and accelerator.is_main_process:
        ckpt_path = cfg.output_dir / f"step_{global_step}"
        accelerator.unwrap_model(model).save_pretrained(str(ckpt_path))
        logger.info("Saved checkpoint to %s", ckpt_path)

accelerator.end_training()
logger.info("Training complete — %d steps", global_step)